# BirdCLEF 2026 — Inference Notebook

**Setup:**
1. Add the `birdclef-model` dataset via **Add Data → Your Datasets**
2. Run all cells → `submission.csv` is saved → click **Submit**

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf

print('TF version:', tf.__version__)
print('Devices:', tf.config.list_physical_devices())

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
COMPETITION_DIR = '/kaggle/input/competitions/birdclef-2026/'
WEIGHTS_PATH    = '/kaggle/input/datasets/pedrogasparfernandes/birdclef-model/best_model_weights.weights.h5'

TEST_DIR        = COMPETITION_DIR + 'test_soundscapes/'
TAXONOMY_PATH   = COMPETITION_DIR + 'taxonomy.csv'
SAMPLE_SUB_PATH = COMPETITION_DIR + 'sample_submission.csv'

# ── Audio config — must match training exactly ────────────────────────────────
SAMPLE_RATE = 32000
DURATION    = 5
N_MELS      = 64
F_MAX       = 16000
BATCH_SIZE  = 32

In [ ]:
# ── Build model and load weights ──────────────────────────────────────────────
# Backbone
inputs_bb = tf.keras.Input(shape=(64, 313, 1))
x_bb      = tf.keras.layers.Concatenate(axis=-1)([inputs_bb, inputs_bb, inputs_bb])
base      = tf.keras.applications.EfficientNetB0(include_top=False, weights=None, pooling='avg')
backbone  = tf.keras.Model(inputs_bb, base(x_bb), name='backbone_efficientnet')

# Head
inputs  = tf.keras.Input(shape=(64, 313, 1))
x       = backbone(inputs, training=False)
x       = tf.keras.layers.Dense(1024, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.3)(x)
x       = tf.keras.layers.BatchNormalization()(x)
x       = tf.keras.layers.Dense(512, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(234, activation='sigmoid')(x)
model   = tf.keras.Model(inputs, outputs)

model.load_weights(WEIGHTS_PATH)
print('Model loaded.')
model.summary()

In [ ]:
# ── Load taxonomy — defines species column order ──────────────────────────────
# Cast to str so column names match the string headers in sample_submission.csv
taxonomy = pd.read_csv(TAXONOMY_PATH)
species  = taxonomy['primary_label'].astype(str).tolist()  # 234 species, must be strings
print(f'Species: {len(species)}')

# Verify against sample submission
sample_sub     = pd.read_csv(SAMPLE_SUB_PATH)
sub_species    = sample_sub.columns[1:].tolist()
assert species == sub_species, f'Species mismatch: taxonomy has {species[:3]}, submission has {sub_species[:3]}'
print('Species order verified against sample_submission.')

In [ ]:
# ── Audio helpers — identical to agent.py ────────────────────────────────────
def audio_to_melspectrogram(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, fmax=F_MAX
    )
    return librosa.power_to_db(mel, ref=np.max)


def extract_windows(filepath):
    """Load a soundscape and return list of (end_second, spectrogram) tuples."""
    try:
        audio, _ = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f'  Failed to load {filepath}: {e}')
        return []

    windows = []
    step    = SAMPLE_RATE * DURATION
    total   = len(audio)

    for start in range(0, total, step):
        chunk = audio[start:start + step]
        if len(chunk) < step:
            chunk = np.pad(chunk, (0, step - len(chunk)))
        mel        = audio_to_melspectrogram(chunk)
        end_second = start // SAMPLE_RATE + DURATION
        windows.append((end_second, mel))

    return windows

In [ ]:
# ── Run inference on all test soundscapes ────────────────────────────────────
# Walk recursively in case files are in subdirectories
test_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(TEST_DIR)
    for f in files if f.endswith('.ogg')
])
print(f'Test files: {len(test_files)}')

rows = []

for i, fpath in enumerate(test_files):
    stem    = os.path.splitext(os.path.basename(fpath))[0]
    windows = extract_windows(fpath)

    if not windows:
        continue

    end_seconds = [w[0] for w in windows]
    specs       = np.array([w[1] for w in windows])[..., np.newaxis]

    for batch_start in range(0, len(specs), BATCH_SIZE):
        batch      = specs[batch_start:batch_start + BATCH_SIZE]
        preds      = model.predict(batch, verbose=0)
        batch_ends = end_seconds[batch_start:batch_start + BATCH_SIZE]

        for end_sec, pred in zip(batch_ends, preds):
            row = {'row_id': f'{stem}_{end_sec}'}
            row.update(dict(zip(species, pred.tolist())))
            rows.append(row)

    if (i + 1) % 10 == 0:
        print(f'  Processed {i + 1}/{len(test_files)} files')

print(f'Total rows: {len(rows)}')

In [ ]:
# ── Build submission ──────────────────────────────────────────────────────────
if rows:
    submission = pd.DataFrame(rows)[['row_id'] + species]
else:
    print('WARNING: no rows — falling back to sample_submission with zero scores')
    submission = sample_sub.copy()
    submission[species] = 0.0

submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved — {len(submission)} rows x {len(submission.columns)} columns')
submission.head(3)